<a href="https://colab.research.google.com/github/vandita1111/Game-Design-Project/blob/main/Copy_of_semantic_similarity_with_tf_hub_universal_encoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2018 The TensorFlow Hub Authors.

Licensed under the Apache License, Version 2.0 (the "License");

In [10]:
# Copyright 2018 The TensorFlow Hub Authors. All Rights Reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# ==============================================================================

# Universal Sentence Encoder


<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://www.tensorflow.org/hub/tutorials/semantic_similarity_with_tf_hub_universal_encoder"><img src="https://www.tensorflow.org/images/tf_logo_32px.png" />View on TensorFlow.org</a>
  </td>
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/tensorflow/docs/blob/master/site/en/hub/tutorials/semantic_similarity_with_tf_hub_universal_encoder.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/tensorflow/docs/blob/master/site/en/hub/tutorials/semantic_similarity_with_tf_hub_universal_encoder.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View on GitHub</a>
  </td>
  <td>
    <a href="https://storage.googleapis.com/tensorflow_docs/docs/site/en/hub/tutorials/semantic_similarity_with_tf_hub_universal_encoder.ipynb"><img src="https://www.tensorflow.org/images/download_logo_32px.png" />Download notebook</a>
  </td>
  <td>
    <a href="https://tfhub.dev/s?q=google%2Funiversal-sentence-encoder%2F4%20OR%20google%2Funiversal-sentence-encoder-large%2F5"><img src="https://www.tensorflow.org/images/hub_logo_32px.png" />See TF Hub models</a>
  </td>
</table>

This notebook illustrates how to access the Universal Sentence Encoder and use it for sentence similarity and sentence classification tasks.

The Universal Sentence Encoder makes getting sentence level embeddings as easy as it has historically been to lookup the embeddings for individual words. The sentence embeddings can then be trivially used to compute sentence level meaning similarity as well as to enable better performance on downstream classification tasks using less supervised training data.


## Setup

This section sets up the environment for access to the Universal Sentence Encoder on TF Hub and provides examples of applying the encoder to words, sentences, and paragraphs.

In [11]:
%%capture
!pip3 install seaborn

More detailed information about installing Tensorflow can be found at [https://www.tensorflow.org/install/](https://www.tensorflow.org/install/).

In [2]:
#@title Load the Universal Sentence Encoder's TF Hub module
from absl import logging

import tensorflow as tf

import tensorflow_hub as hub
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re
import seaborn as sns

module_url = "https://tfhub.dev/google/universal-sentence-encoder/4" #@param ["https://tfhub.dev/google/universal-sentence-encoder/4", "https://tfhub.dev/google/universal-sentence-encoder-large/5"]
model = hub.load(module_url)
print ("module %s loaded" % module_url)
def embed(input):
  return model(input)

module https://tfhub.dev/google/universal-sentence-encoder/4 loaded


In [3]:
from google.colab import files
uploaded = files.upload()


Saving Sherlock_Segments.csv to Sherlock_Segments.csv
Saving Sherlock_Segments.xlsx to Sherlock_Segments.xlsx
Saving subject_recalls.csv to subject_recalls.csv


In [4]:
import pandas as pd
import tensorflow_hub as hub
from absl import logging

import tensorflow as tf

import tensorflow_hub as hub
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re
import seaborn as sns

module_url = "https://tfhub.dev/google/universal-sentence-encoder/4"
model = hub.load(module_url)
print ("module %s loaded" % module_url)
def embed(input):
  return model(input)
subject_recalls = pd.read_csv("subject_recalls.csv")
sherlock_segments = pd.read_csv("Sherlock_Segments.csv")


module https://tfhub.dev/google/universal-sentence-encoder/4 loaded


In [27]:
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict

subject_recalls = pd.read_csv("subject_recalls.csv")
sherlock_segments = pd.read_csv("Sherlock_Segments.csv")
#print("Recalls columns:", subject_recalls.columns.tolist())
#print("Segments columns:", sherlock_segments.columns.tolist())

# Filter both dataframes to ensure we have valid text data
recall_texts = subject_recalls.dropna(subset=['Text'])
recall_events = subject_recalls.dropna(subset=['Event'])
recall_subjects = subject_recalls.dropna(subset=['Subject'])

sherlock_description = sherlock_segments.dropna(subset=['scene_detail'])
sherlock_events = sherlock_segments.dropna(subset=['event_number'])

# write to list
recall_texts = recall_texts['Text'].tolist() #all text recalled by participants
recall_events = recall_events['Event'].tolist() #all events recalled
recall_subjects = recall_subjects['Subject'].tolist() #subject ids corresponding to text

sherlock_description = sherlock_description['scene_detail'].tolist() #event descriptions
sherlock_events = sherlock_events['event_number'].tolist() #all possible events

# Generate embeddings using the Universal Sentence Encoder

subjects = set(recall_subjects) #set of subjects
subjects = sorted(list(subjects)) #convert to list
subject_indexes = subjects.copy() #copy list
#print(subjects)
for i in range(0, len(subjects)):
  curr_subject = subjects[i] #iterate through subjects
  if i < len(subjects)-1:
    next_subject = subjects[i+1]
    index_first = recall_subjects.index(curr_subject) #get the first index of recall for this subject
    index_last = recall_subjects.index(next_subject) #get the last index
  else:
    index_first = recall_subjects.index(curr_subject)
    index_last = len(recall_subjects) #+1
  subject_indexes[i] = list(range(index_first, index_last+1))
  #subjects[i] = subject_indexes
#print(subject_indexes)
#print(len(subject_indexes))
#print(recall_texts[subject_indexes[-1][-1]])

similarity_scores = []
num_recalls = np.zeros((len(subjects), len(sherlock_events)))

#EDIT THIS SO IT USES SUBJECTS AS DEFINED DYNAMICALLY!
for y in range(0, len(sherlock_events)): #for each event in sherlock (~34)
  curr_event = int(sherlock_events[y])
  curr_description = [sherlock_description[y]]

  sherlock_embedding = model(curr_description) #generate embedding for this event

  for i in range(0,len(subjects)): #for each subject (18)
    curr_subject = subject_indexes[i] #iterate through each subject (1-18)
    #print(curr_subject)
    recall_curr_event = []

    for j in range(0, len(curr_subject)): #for each index of this subject
      events_at_index = recall_events[curr_subject[j]] #these are the events at this index
      events_at_index = events_at_index.split(', ') #split by commas
      #print(events_at_index)

      if str(curr_event) in events_at_index: #if the current event is in this cell for this subject
        recall_curr_event.append(recall_texts[curr_subject[j]]) #append the recall at this index
        num_recalls[i, y] = 1 # event is recalled
    print(i, curr_event, recall_curr_event)

    if recall_curr_event:
      #print("this should be the all the recall of event 2 by pt1 ", recall_curr_event)
      recall_embedding = model(recall_curr_event) #generate embedding for this recalled event for this subject
      similarity_score = cosine_similarity(sherlock_embedding, recall_embedding)
      similarity_scores.append([i+1, similarity_score, curr_event])
#print(similarity_scores) #prints for each event a similarity score for every subject

optimal_no_recalls = len(sherlock_events)*len(subjects)
events_recalled_pcnt = (len(similarity_scores)/optimal_no_recalls)
num_recalls = np.sum(num_recalls, axis=1)
max_recall = max(num_recalls)
min_recall = min(num_recalls)

print("STATS:")
print("total number of events shown: ", len(sherlock_events))
print("total number of subjects: ", len(subjects))
print("number of events recalled by all subjects: ", len(similarity_scores), "/", optimal_no_recalls)
print("total percentage of events recalled: ", round(100*events_recalled_pcnt, 2))
print("most events recalled by a subject: ", max_recall)
print("least events recalled by a subject: ", min_recall)
print("avg number of recalled events: ", round((events_recalled_pcnt * optimal_no_recalls)/len(subjects), 2), "/", len(sherlock_events))




0 2 ['So it starts off with John Watson. He was asleep', 'and he was having like flashbacks of being in the in the military in the army.', 'So the clip started out with some war action scenes and people were shooting at each other and then it looked like it was a flashback ']
1 2 ['So the clip started out with some war action scenes and people were shooting at each other and then it looked like it was a flashback ', 'So, in the first scene we see John Watson.']
2 2 ['So, in the first scene we see John Watson.', 'Mostly, he sees visuals of people, war and stuff of that sort', "The movie starts with a flashback to a war that John Watson was in. It's a nightmare"]
3 2 ["The movie starts with a flashback to a war that John Watson was in. It's a nightmare"]
4 2 ['So the movie starts with a man he gets a dream about a war where he got shot']
5 2 [' So at the beginning of the clip we saw a man sleeping and he was having a nightmare. He saw in his dream it was a war going on and a lot of soldi

In [22]:
print(similarity_scores)


[[1, array([[0.12538686, 0.25387388, 0.31285366]], dtype=float32), 2], [2, array([[0.31285357, 0.17891616]], dtype=float32), 2], [3, array([[0.1789161 , 0.19204131, 0.24310249]], dtype=float32), 2], [4, array([[0.24310252]], dtype=float32), 2], [5, array([[0.28952914]], dtype=float32), 2], [6, array([[0.38990837, 0.10140786]], dtype=float32), 2], [1, array([[0.13525537, 0.19111782]], dtype=float32), 3], [2, array([[ 0.22691104, -0.00304846]], dtype=float32), 3], [3, array([[-0.00304852,  0.06967401,  0.13120447]], dtype=float32), 3], [4, array([[0.5178678 , 0.01975347]], dtype=float32), 3], [5, array([[0.20061392]], dtype=float32), 3], [6, array([[0.13263057]], dtype=float32), 3], [1, array([[0.09058749]], dtype=float32), 4], [2, array([[0.54058397]], dtype=float32), 4], [3, array([[0.04472422]], dtype=float32), 4], [4, array([[0.4444784]], dtype=float32), 4], [5, array([[0.11402871]], dtype=float32), 4], [6, array([[0.23036632]], dtype=float32), 4], [2, array([[0.40424138]], dtype=flo

In [ ]:
n

In [54]:
import numpy as np

subjects = [1, 2, 3, 4, 5, 6]
events = list(range(1,35))


recall_event = script_df.dropna(subset=['Event'])
recall_subject = script_df.dropna(subset=['Subject'])
annot_event = annotations_df.dropna(subset=['Scene_Segments'])

recall_event = df_filtered['Event'].tolist()
recall_subject = df_filtered['Subject'].tolist()
# Use annot_filtered here instead of df_filtered
annot_event = annot_filtered['Scene_Segments'].tolist()
print(annot_event)

for i in range(0, len(recall_subject)): #iterate through subjects
  if recall_subject[i + 1] == recall_subject[i]: #if subject is same as last
    print(recall_subject[i+1])
  for j in range(0, len(recall_event)): #iterate through recalled events
    if recall_event[i] == j: #if recall_event for this subject = curr event
      print(recall_event[i])

[1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 7, 7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 9, 9, 9, 9, 10, 10, 10, 10, 10, 10, 10, 10, 11, 11, 11, 11, 11, 11, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 17, 17, 17, 17, 17, 17, 17, 17, 17, 18, 19, 19, 19, 19, 19, 19,

IndexError: list index out of range

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

# Convert tensors to numpy arrays
recall_np = recall_embeddings.numpy()
annot_np = annot_embeddings.numpy()

# Create DataFrames
# Each row is an embedding vector (512 dimensions)
df_recalls = pd.DataFrame(recall_np)
df_annots = pd.DataFrame(annot_np)

# Save to a single Excel file with multiple sheets
with pd.ExcelWriter('embeddings.xlsx') as writer:
    df_recalls.to_excel(writer, sheet_name='Recall_Embeddings', index=False)
    df_annots.to_excel(writer, sheet_name='Annotation_Embeddings', index=False)

# Download the file to local computer
files.download('embeddings.xlsx')

print('Excel file created and download triggered.')

In [ ]:
#@title Compute a representation for each message, showing various lengths supported.
word = "Elephant"
sentence = "I am a sentence for which I would like to get its embedding."
paragraph = (
    "Universal Sentence Encoder embeddings also support short paragraphs. "
    "There is no hard limit on how long the paragraph is. Roughly, the longer "
    "the more 'diluted' the embedding will be.")
messages = [word, sentence, paragraph]

# Reduce logging output.
logging.set_verbosity(logging.ERROR)

message_embeddings = embed(messages)

for i, message_embedding in enumerate(np.array(message_embeddings).tolist()):
  print("Message: {}".format(messages[i]))
  print("Embedding size: {}".format(len(message_embedding)))
  message_embedding_snippet = ", ".join(
      (str(x) for x in message_embedding[:3]))
  print("Embedding: [{}, ...]\n".format(message_embedding_snippet))

# Semantic Textual Similarity Task Example

The embeddings produced by the Universal Sentence Encoder are approximately normalized. The semantic similarity of two sentences can be trivially computed as the inner product of the encodings.

In [ ]:
def plot_similarity(labels, features, rotation):
  corr = np.inner(features, features)
  sns.set(font_scale=1.2)
  g = sns.heatmap(
      corr,
      xticklabels=labels,
      yticklabels=labels,
      vmin=0,
      vmax=1,
      cmap="YlOrRd")
  g.set_xticklabels(labels, rotation=rotation)
  g.set_title("Semantic Textual Similarity")

def run_and_plot(messages_):
  message_embeddings_ = embed(messages_)
  plot_similarity(messages_, message_embeddings_, 90)

## Similarity Visualized
Here we show the similarity in a heat map. The final graph is a 9x9 matrix where each entry `[i, j]` is colored based on the inner product of the encodings for sentence `i` and `j`.

In [ ]:
messages = [
    # Smartphones
    "I like my phone",
    "My phone is not good.",
    "Your cellphone looks great.",

    # Weather
    "Will it snow tomorrow?",
    "Recently a lot of hurricanes have hit the US",
    "Global warming is real",

    # Food and health
    "An apple a day, keeps the doctors away",
    "Eating strawberries is healthy",
    "Is paleo better than keto?",

    # Asking about age
    "How old are you?",
    "what is your age?",
]

run_and_plot(messages)


## Evaluation: STS (Semantic Textual Similarity) Benchmark

The [**STS Benchmark**](https://ixa2.si.ehu.eus/stswiki/stswiki.html#STS_benchmark) provides an intrinsic evaluation of the degree to which similarity scores computed using sentence embeddings align with human judgements. The benchmark requires systems to return similarity scores for a diverse selection of sentence pairs. [Pearson correlation](https://en.wikipedia.org/wiki/Pearson_correlation_coefficient) is then used to evaluate the quality of the machine similarity scores against human judgements.

### Download data

In [ ]:
import pandas
import scipy
import math
import csv

sts_dataset = tf.keras.utils.get_file(
    fname="Stsbenchmark.tar.gz",
    origin="http://ixa2.si.ehu.es/stswiki/images/4/48/Stsbenchmark.tar.gz",
    extract=True)
sts_dev = pandas.read_table(
    os.path.join(os.path.dirname(sts_dataset), "stsbenchmark", "sts-dev.csv"),
    skip_blank_lines=True,
    usecols=[4, 5, 6],
    names=["sim", "sent_1", "sent_2"])
sts_test = pandas.read_table(
    os.path.join(
        os.path.dirname(sts_dataset), "stsbenchmark", "sts-test.csv"),
    quoting=csv.QUOTE_NONE,
    skip_blank_lines=True,
    usecols=[4, 5, 6],
    names=["sim", "sent_1", "sent_2"])
# cleanup some NaN values in sts_dev
sts_dev = sts_dev[[isinstance(s, str) for s in sts_dev['sent_2']]]

### Evaluate Sentence Embeddings

In [ ]:
sts_data = sts_dev #@param ["sts_dev", "sts_test"] {type:"raw"}

def run_sts_benchmark(batch):
  sts_encode1 = tf.nn.l2_normalize(embed(tf.constant(batch['sent_1'].tolist())), axis=1)
  sts_encode2 = tf.nn.l2_normalize(embed(tf.constant(batch['sent_2'].tolist())), axis=1)
  cosine_similarities = tf.reduce_sum(tf.multiply(sts_encode1, sts_encode2), axis=1)
  clip_cosine_similarities = tf.clip_by_value(cosine_similarities, -1.0, 1.0)
  scores = 1.0 - tf.acos(clip_cosine_similarities) / math.pi
  """Returns the similarity scores"""
  return scores

dev_scores = sts_data['sim'].tolist()
scores = []
for batch in np.array_split(sts_data, 10):
  scores.extend(run_sts_benchmark(batch))

pearson_correlation = scipy.stats.pearsonr(scores, dev_scores)
print('Pearson correlation coefficient = {0}\np-value = {1}'.format(
    pearson_correlation[0], pearson_correlation[1]))